# Drift Monitoring — Churn Feature Distribution Shift

This notebook performs:
1. **Population Stability Index (PSI)** — compares current vs. baseline feature distributions
2. **Kolmogorov-Smirnov test** — non-parametric distribution shift detection
3. **Alert thresholds**: PSI > 0.10 → moderate; PSI > 0.25 → severe (trigger retraining)
4. **Time-series view**: rolling weekly feature means to spot trend drift

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, str(Path().resolve().parent))
from src.config import ARTIFACTS_DIR, PROCESSED_DATA_DIR

FEATURE_STORE = PROCESSED_DATA_DIR / 'feature_store'
CHURN_DIR     = ARTIFACTS_DIR / 'churn'

FEATURE_COLS = [
    'total_ratings', 'avg_rating', 'std_rating', 'rating_sessions',
    'days_since_first', 'days_since_last', 'avg_session_gap_days',
    'pct_high_rating', 'genre_diversity',
]

print('Config loaded')

## 1. Load Data

In [ ]:
feat_path = FEATURE_STORE / 'user_churn_features.parquet'
baseline_path = CHURN_DIR / 'feature_baseline.parquet'

if not feat_path.exists():
    print('Feature store not found — running feature engineering...')
    from src.churn.feature_engineering import build_churn_features, save_feature_store
    current_df = build_churn_features()
    save_feature_store(current_df)
else:
    current_df = pd.read_parquet(str(feat_path))

print(f'Current features: {len(current_df):,} users')
print(current_df[FEATURE_COLS].describe().round(3))

## 2. Population Stability Index (PSI)

In [ ]:
def compute_psi(expected: np.ndarray, actual: np.ndarray, n_bins: int = 10) -> float:
    """Population Stability Index."""
    eps = 1e-8
    all_vals = np.concatenate([expected, actual])
    bin_edges = np.percentile(all_vals, np.linspace(0, 100, n_bins + 1))
    bin_edges = np.unique(bin_edges)  # handle duplicate edges
    if len(bin_edges) < 3:
        return 0.0
    exp_cnt, _ = np.histogram(expected, bins=bin_edges)
    act_cnt, _ = np.histogram(actual,   bins=bin_edges)
    exp_pct = (exp_cnt / len(expected)) + eps
    act_pct = (act_cnt / len(actual))   + eps
    return float(np.sum((act_pct - exp_pct) * np.log(act_pct / exp_pct)))

def drift_label(psi: float) -> str:
    if psi > 0.25: return '🔴 SEVERE'
    if psi > 0.10: return '🟡 MODERATE'
    return '🟢 OK'

# Simulate baseline = slightly older distribution (shift days_since_last by +5 days)
if not baseline_path.exists():
    baseline_df = current_df.copy()
    baseline_df['days_since_last'] = baseline_df['days_since_last'] + 5
    print('Baseline not found — using simulated baseline (current + 5-day shift)')
else:
    baseline_df = pd.read_parquet(str(baseline_path))

print(f'Baseline features: {len(baseline_df):,} users')

psi_results = []
for col in FEATURE_COLS:
    if col not in baseline_df.columns:
        continue
    exp = baseline_df[col].dropna().values
    act = current_df[col].dropna().values
    psi_val = compute_psi(exp, act)
    ks_stat, ks_p = stats.ks_2samp(exp, act)
    psi_results.append({
        'feature':     col,
        'psi':         round(psi_val, 4),
        'ks_stat':     round(float(ks_stat), 4),
        'ks_p_value':  round(float(ks_p), 4),
        'drift_level': drift_label(psi_val),
    })

drift_df = pd.DataFrame(psi_results).sort_values('psi', ascending=False)
print(drift_df.to_string(index=False))

## 3. Visual Distribution Comparison

In [ ]:
import matplotlib.pyplot as plt

top_features = drift_df.nlargest(4, 'psi')['feature'].tolist()
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for ax, feat in zip(axes, top_features):
    baseline_vals = baseline_df[feat].dropna()
    current_vals  = current_df[feat].dropna()
    psi_v = drift_df.loc[drift_df['feature'] == feat, 'psi'].iloc[0]
    
    ax.hist(baseline_vals, bins=30, alpha=0.6, label='Baseline', color='steelblue', density=True)
    ax.hist(current_vals,  bins=30, alpha=0.6, label='Current',  color='coral',     density=True)
    ax.set_title(f'{feat}\nPSI={psi_v:.3f}', fontsize=10)
    ax.legend(fontsize=8)
    ax.set_xlabel(feat, fontsize=8)
    ax.set_ylabel('Density', fontsize=8)

plt.suptitle('Feature Distribution: Baseline vs. Current', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(str(CHURN_DIR / 'drift_distributions.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Distribution plot saved')

## 4. Alert Summary

In [ ]:
severe   = drift_df[drift_df['psi'] > 0.25]
moderate = drift_df[(drift_df['psi'] > 0.10) & (drift_df['psi'] <= 0.25)]
ok       = drift_df[drift_df['psi'] <= 0.10]

print('=' * 50)
print('DRIFT ALERT SUMMARY')
print('=' * 50)
print(f'  🟢 OK       : {len(ok)} feature(s)')
print(f'  🟡 MODERATE : {len(moderate)} feature(s)')
print(f'  🔴 SEVERE   : {len(severe)} feature(s)')
if not severe.empty:
    print(f'\n  ⚠️  Retraining recommended for features: {severe["feature"].tolist()}')
else:
    print('\n  ✓ No severe drift — model is stable')